# Notebook 4: Fusion-Layer Attacks

Demonstrate all 9 fusion attack types targeting the JIPDA tracker.

**Attacks:**
- Existence Suppression
- Association Confusion
- Cross-Sensor Consistency
- False Track Injection
- Sensor DoS
- Track Deletion
- Track Swap
- Stealthy Degradation
- Track Merge Manipulation

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader
from attacks.fusion_attacks import FusionAttacker, FusionAttackType

%matplotlib inline

## 4.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()
ground_truth = loader.load_ground_truth()

print(f"Sensors: {list(detections.keys())}")
print(f"Targets: {list(ground_truth.keys())}")

## 4.2 Initialize Fusion Attacker

In [ ]:
attacker = FusionAttacker()
print("Available fusion attacks:")
for a in FusionAttackType:
    print(f"  - {a.name}")

## 4.3 Run All Fusion Attacks

In [ ]:
results = {}
for attack_type in FusionAttackType:
    attacked = attacker.attack_scenario(detections, attack_type, ground_truth)
    results[attack_type.name] = attacked
    
    # Count total detections per sensor
    counts = {sid: len(df) for sid, df in attacked.items()}
    print(f"{attack_type.name:25s}: {counts}")

## 4.4 Visualize Fusion Attack Impact

In [ ]:
# Select key attacks to visualize
key_attacks = [
    'EXISTENCE_SUPPRESSION',
    'FALSE_TRACK_INJECTION',
    'SENSOR_DOS',
    'TRACK_DELETION',
    'TRACK_SWAP',
    'STEALTHY_DEGRADATION',
    'TRACK_MERGE_MANIPULATION'
]

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

# Benign baseline
ax = axes[0]
for sid, df in detections.items():
    ax.scatter(df['x_piren'], df['y_piren'], s=2, alpha=0.3, label=f'Sensor {sid}')
for tid, gt in ground_truth.items():
    ax.plot(gt['x_piren'], gt['y_piren'], 'k-', linewidth=1.5)
ax.set_title('BENIGN (Baseline)')
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.legend()
ax.grid(True)
ax.set_aspect('equal')

for idx, attack_name in enumerate(key_attacks, 1):
    ax = axes[idx]
    attacked = results[attack_name]
    
    for sid, df in attacked.items():
        ax.scatter(df['x_piren'], df['y_piren'], s=2, alpha=0.3)
    for tid, gt in ground_truth.items():
        ax.plot(gt['x_piren'], gt['y_piren'], 'k-', linewidth=1.5)
    
    ax.set_title(attack_name.replace('_', ' '))
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')

# Hide unused subplot
axes[-1].axis('off')

plt.suptitle('Fusion Attacks Impact', fontsize=16)
plt.tight_layout()
plt.show()

## 4.5 Track-Oriented Attacks Detail

In [ ]:
# Focus on track deletion, swap, merge
track_attacks = ['TRACK_DELETION', 'TRACK_SWAP', 'TRACK_MERGE_MANIPULATION']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, attack_name in zip(axes, track_attacks):
    attacked = results[attack_name]
    
    # Plot all sensor detections
    for sid, df in attacked.items():
        ax.scatter(df['x_piren'], df['y_piren'], s=3, alpha=0.4, label=f'Sensor {sid}')
    
    # Plot ground truth with different styles
    colors = ['red', 'green', 'blue', 'orange']
    for idx, (tid, gt) in enumerate(ground_truth.items()):
        ax.plot(gt['x_piren'], gt['y_piren'], '--', color=colors[idx % len(colors)], 
                linewidth=2, label=f'Target {tid}')
    
    ax.set_title(attack_name.replace('_', ' '))
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend(fontsize=8)
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Track-Oriented Fusion Attacks', fontsize=14)
plt.tight_layout()
plt.show()

## 4.6 Stealthy Degradation Over Time

In [ ]:
# Show how stealthy degradation progresses
attacked = results['STEALTHY_DEGRADATION']

# Get time bins
all_times = []
for df in attacked.values():
    all_times.extend(df['time'].tolist())

t_min, t_max = min(all_times), max(all_times)
time_bins = np.linspace(t_min, t_max, 5)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx in range(len(time_bins) - 1):
    ax = axes[idx]
    t_start, t_end = time_bins[idx], time_bins[idx + 1]
    
    # Plot detections in this time window
    for sid, df in attacked.items():
        mask = (df['time'] >= t_start) & (df['time'] < t_end)
        subset = df[mask]
        ax.scatter(subset['x_piren'], subset['y_piren'], s=5, alpha=0.5, label=f'Sensor {sid}')
    
    for tid, gt in ground_truth.items():
        mask = (gt['time'] >= t_start) & (gt['time'] < t_end)
        gt_sub = gt[mask]
        if len(gt_sub) > 0:
            ax.plot(gt_sub['x_piren'], gt_sub['y_piren'], 'k-', linewidth=2)
    
    progress = (idx / (len(time_bins) - 2)) * 100
    ax.set_title(f'Time: {t_start:.0f}-{t_end:.0f}s (Progress: {progress:.0f}%)')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend()
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Stealthy Degradation Over Time', fontsize=14)
plt.tight_layout()
plt.show()